
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 강의 - MLflow를 활용한 Databricks 기반 에이전트 구축

MLflow는 하나의 통합 플랫폼 내에서 엔드투엔드 추적, 관찰 가능성(observability), 평가 기능을 제공하여 생성형 AI(GenAI) 애플리케이션의 성능을 향상시킵니다. 따라서 개발, 평가, 배포부터 실제 운영 모니터링에 이르기까지 AI 에이전트의 전체 라이프사이클을 지원합니다. 본 강의에서는 초기 프로토타입 제작부터 운영 배포 및 지속적인 관찰에 이르기까지, 에이전트 개발 라이프사이클에서 MLflow의 포괄적인 기능이 왜 필수적인 요소인지 살펴봅니다.

## 학습 목표

_본 강의를 마치면 여러분은 다음을 수행할 수 있게 됩니다_

- MLflow의 실험 추적(experiment tracking) 기능이 에이전트의 반복적인 개발을 어떻게 지원하는지 설명할 수 있습니다.
- 포괄적인 에이전트 관찰 가능성을 제공하는 데 있어 MLflow 트레이싱(tracing)과 태깅(tagging)의 역할을 기술할 수 있습니다.
- MLflow 모델 레지스트리(model registry)를 통해 재현 가능하고 거버넌스가 확보된 에이전트 배포를 수행하는 방법을 파악할 수 있습니다.
- 엔터프라이즈급 에이전트 관리를 위한 MLflow와 Unity Catalog 통합의 이점을 분석할 수 있습니다.

> 본 강의는 에이전트를 위한 MLflow의 기본 개념을 소개하기 위한 것입니다. MLflow의 모든 구성 요소를 완전하게 설명하는 것을 목적으로 하지 않으며, [Databricks 관리 MLflow](https://www.databricks.com/product/managed-mlflow)에 초점을 맞추고 있습니다. OSS MLflow에 관한 문서는 [여기](https://mlflow.org/)에서 더 읽어보실 수 있습니다.

## A. 에이전트 개발의 과제

실제 서비스 환경에 바로 적용할 수 있는(production-ready) AI 에이전트를 구축하는 일은 기존의 머신러닝 워크플로만으로는 해결하기 어려운 까다로운 과제들을 동반합니다. 이러한 난제들을 이해하고 나면, 왜 Databricks 환경의 MLflow가 에이전트의 전체 라이프사이클을 아우르는 종합 플랫폼으로 자리 잡게 되었는지 그 이유를 명확히 알 수 있습니다.

### A1. 에이전트 시스템의 복잡성

AI 에이전트는 다음과 같은 몇 가지 핵심적인 측면에서 기존 머신러닝 모델과 근본적으로 다릅니다.

- **다단계 추론(Multi-step reasoning)**: 에이전트는 여러 단계에 걸쳐 계획 수립, 도구 활용, 의사결정 등을 포함하는 복잡하고 유기적인 다회차(multi-turn) 상호작용을 수행합니다.
- **동적 행동(Dynamic behavior)**: 고정된 모델과 달리, 에이전트는 문맥이나 사용할 수 있는 도구, 그리고 이전 대화 기록에 따라 각기 다른 행동을 보일 수 있습니다.
- **도구 통합(Tool integration)**: 에이전트는 주어진 작업을 완수하기 위해 외부 시스템, API, 데이터 소스 등과 매끄럽게 연동되어야 합니다.
- **대화 문맥 유지(Conversational context)**: 여러 차례 오가는 대화 속에서 상태와 문맥을 유지해야 하므로, 배포 및 모니터링의 복잡성이 더욱 커집니다.

이러한 특성들 때문에 에이전트는 개발, 테스트, 운영 모니터링 전반에 걸쳐 고유한 요구사항을 가지며, 이는 기존의 머신러닝 플랫폼들이 처음에 해결하고자 했던 범위를 넘어섭니다.

### A2. 관측 가능성(Observability) 요구사항

에이전트의 관측 가능성은 단순히 기존의 모델 모니터링 수준을 훨씬 뛰어넘습니다.

- **실행 추적(Execution tracing)**: 에이전트가 어떤 도구를 왜 호출했는지를 포함하여, 단계별 추론 과정을 상세히 파악해야 합니다.
- **성능 분석(Performance analysis)**: 잡한 다단계 워크플로 전반에 걸쳐 지연 시간(latency), 토큰 사용량, 비용 등을 추적해야 합니다.
- **품질 평가(Quality assessment)**: 최종 출력 결과뿐만 아니라, 중간 추론 단계와 도구 활용 패턴까지 모두 평가해야 합니다.
- **오류 진단(Error diagnosis)**: 여러 단계로 이루어진 프로세스 중 어느 지점에서 실패가 발생했는지 찾아내고 그 근본 원인을 파악해야 합니다.

적절한 관측 가능성이 확보되지 않으면, 특히 실제 운영 환경에서 에이전트의 동작을 디버깅하는 것은 거의 불가능에 가깝습니다.

### A3. 거버넌스 및 재현성 과제

에이전트의 엔터프라이즈 배포는 강력한 거버넌스 역량이 필요합니다:

- **버전 관리(Version management)**: 에이전트 로직, 프롬프트, 도구, 설정 등의 변경 사항을 추적해야 합니다.
- **재현성(Reproducibility)**: 개발, 스테이징, 운영 환경 전반에서 일관된 동작을 보장해야 합니다.
- **접근 제어(Access control)**: 다양한 에이전트 버전을 배포, 수정, 접근할 수 있는 권한을 관리해야 합니다.
- **감사 추적(Audit trails)**: 컴플라이언스 준수 및 디버깅을 위해 에이전트의 동작에 대한 전체 기록을 유지해야 합니다.
- **AI 가드레일(AI Guardrails)**: 사용자가 모델 서빙 엔드포인트 수준에서 데이터 컴플라이언스를 설정하고 강제할 수 있도록 지원합니다.(자세한 내용은 [여기](https://docs.databricks.com/aws/en/ai-gateway/#ai-guardrails) 읽기)

이러한 요구사항들을 충족하려면 엔터프라이즈급 거버넌스 기능을 갖추고 에이전트 라이프사이클 전체를 관리할 수 있는 플랫폼이 반드시 필요합니다.

## B. Databricks 기반 MLflow
여기에서는 MLflow를 조금 더 자세히 분석하여, 앞 섹션에서 제기된 문제들을 MLflow가 어떻게 해결하는지 이해해 보겠습니다.

### B1. 트레이싱(Tracing)이 필요한 이유

트레이싱이 왜 필요한지(그리고 그것이 실제로 무엇인지) 이해하려면, 우선 전통적인 머신러닝 추론 방식을 이해해야 합니다.

### 전통적인 ML 추론 (요청/응답)
머신러닝의 일반적인 추론 흐름은 다음과 같은 상위 단계로 이루어집니다.
1. 클라이언트가 서빙 엔드포인트의 요청 핸들러(Request Handler)로 입력 요청을 보냅니다.
1. 핸들러는 추론을 위해 요청을 모델에 전달합니다.
1. 모델의 출력 결과가 핸들러를 거쳐 클라이언트에게 반환됩니다.

이러한 기본적인 시나리오에서 투명하게 들여다볼 수 있는 구성 요소는 대개 입력과 출력뿐인 경우가 많습니다.

에이전트 시대에 안정적인 운영을 하려면 **서버 측 투명성, 지연 시간/비용 지표, API 로깅** 같은 프로세스도 파악할 수 있어야 합니다(ML 워크로드에는 필요하지 않더라도 말이죠). 이러한 기능들은 운영 환경의 원격 측정(Telemetry) 및 거버넌스를 위해 **Databricks Model Serving** 과 **AI Gateway** 에서 기본적으로 제공됩니다. Databricks는 **관리형 MLflow**를 호스팅하므로, 트래킹 URI를 `databricks`로 설정하면 보안, 안정성, 검색 기능, UI가 내장된 작업 공간(Workspace)에 트레이스(Trace)가 자동으로 기록됩니다. 또한, [**Mosaic AI Agent Framework**](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent)를 통해 배포하면 실시간 트레이싱이 자동으로 통합되며, 운영 트래픽에 대한 모니터링과 Review App 기능을 활성화할 수 있습니다.

### 에이전트에게는 더 깊은 인사이트가 필요
- 에이전트는 여러 중간 단계(예: 정보 검색, 도구 사용, LLM 호출 등)를 수행하므로, 문제를 디버깅하고 품질을 개선하려면 각 단계와 해당 단계의 입력/출력, 그리고 단계별 지연 시간 및 토큰 사용량을 확인할 수 있어야 합니다.
- **MLflow Tracing** 은 지원되는 라이브러리(OpenAI SDK, LangChain/LangGraph, DSPy 등)에 대해 이러한 과정을 **트레이스(Trace)** 와 **스팬(Span)** 으로 자동 캡처하며, 개발부터 운영 단계까지 이를 분석할 수 있는 UI와 API를 제공합니다.
- **트레이싱** 시스템에서 수행되는 단일 작업을 **스팬(Span)** 이라고 합니다. 스팬은 작업 단위별로 시작 및 종료 시점과 함께 메타데이터, 입력, 출력을 기록합니다.
> MLflow 스팬은 [OpenTelemetry 표준](https://opentelemetry.io/docs/concepts/signals/traces/)을 따르므로, 추가 정보(예: 토큰 수)는 커스텀 필드가 아니라 스팬의 키-값 속성(Attributes)에 저장해야 합니다.

### B2. 에이전트를 위한 트레이싱
이제 에이전트 기반 시스템 개발의 어려움과 에이전트에게 왜 작업 단위별로 더 깊은 인사이트가 필요한지 이해했으므로, **트레이스(Trace)** 가 무엇인지 정의할 준비가 되었습니다. GenAI 애플리케이션 관점에서 **트레이스(Trace)** 는 DAG(방향성 비순환 그래프) 구조로 배열된 스팬들의 집합이며, 여기서 각 스팬은 단일 작업을 나타냅니다. 이 단일 작업은 함수 호출이나 데이터베이스 쿼리 같은 것이 될 수 있습니다.

예를 들어, 3개의 UC(Unity Catalog) 도구에 연결된 에이전트를 개발하고 있다고 가정해 보겠습니다. 실행 속도가 느려지는 것을 감지했지만 무엇이 문제인지 확실치 않은 상황입니다. 이때 Databricks의 MLflow 인터페이스를 활용하면 다음과 같이 문제를 해결할 수 있습니다.
- 각 추론 단계에 사용된 구체적인 파운데이션 모델(Foundation Model) API 확인
- 에이전트에 사용된 시스템 프롬프트(있는 경우) 확인
- 도구가 호출되었는지 여부, 호출된 순서, 그리고 해당 도구들의 입력/출력 확인
- 각 단계별 에이전트의 추론 과정(Reasoning) 파악
- 어떤 도구를 실행하는 데 시간이 가장 오래 걸렸는지 확인하기 위한 지연 시간 파악 (예: 최적화된 SQL 쿼리를 작성하는 데 유용함)
- 스팬 및 트레이스(취합됨)별로 노출되는 토큰 사용량 확인

### B3. 계층적 스팬 구조

MLflow는 에이전트의 실행 과정을 반영하는 계층적 스팬 구조를 사용하여 트레이스 데이터를 구성합니다. 전체 요청이나 워크플로우를 나타내는 단일 루트 스팬(Root Span)에서 시작하여, 각 하위 단계에 대한 하위 스팬(Child Span)이 중첩되는 방식입니다.

- **부모 스팬**: "사용자 요청 처리"와 같은 상위 수준의 작업
- **자식 스팬**: "검색 도구 호출" 또는 "응답 생성"과 같은 상세 단계
- **스팬 관계**: 실행 흐름을 보여주는 명확한 부모-자식 관계(애플리케이션의 실행 계획과 일치해야 함)
- **스팬 유형**: 더 효율적인 조직화를 위한 스팬 분류(`TOOL`, `CHAT_MODEL`, `RETRIEVER` 등)

![tracing-example.png](../Includes/images/tracing-example.png "tracing-example.png")
<p>
<em>
부모-자식 스팬, 관계, 스팬 유형 및 사용된 모델을 보여주는 트레이스 예시.
</em>
</p>

### B4. 커스텀 트레이싱 및 태깅

MLflow는 커스텀 트레이싱 요구사항을 위한 유연한 API도 제공합니다 (이는 데모에서 살펴보겠습니다).

- **커스텀 트레이싱**: [`@mlflow.trace`](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/app-instrumentation/manual-tracing/fluent-apis#decorator) 데코레이터를 사용하면 거의 추가 작업 없이 어떤 함수든 트레이싱되는 스팬으로 전환할 수 있습니다. 이를 적용하면 코드를 계측(Instrument)할 수 있는 가볍고도 강력한 방법이 제공됩니다.
- MLflow는 **트레이싱** 된 함수 간의 부모-자식 관계를 자동으로 유추하여 자동 트레이싱 통합 기능과 완전한 호환성을 보장합니다.
- 함수 내에서 발생하는 모든 예외(Exception)는 캡처되어 **스팬 이벤트** 로 로그에 기록됩니다.
- 추가 설정 없이도 함수의 이름, 입력, 출력, 실행 시간이 기록됩니다.
- `mlflow.openai.autolog`와 같은 자동 트레이싱 기능과 함께 매끄럽게 작동합니다.
- 다음 인자(Arguments)를 허용합니다:
- `name`: 기본값(데코레이터가 적용된 함수의 이름) 대신 스팬 이름을 직접 지정하는 파라미터입니다.
- `span_type`: 스팬의 유형을 설정하는 파라미터입니다. 내장된 [Span Types](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/data-model#span-types) 중 하나를 선택하거나 문자열로 설정할 수 있습니다. 
- `attributes`: 스팬에 커스텀 속성을 추가하기 위한 파라미터입니다.

> **함수 유형 고려사항**
데코레이터를 사용할 `@mlflow.trace` 때 기능 유형과 지원되는 의존성 전체 목록을 뷰하려면 [이 문서](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/app-instrumentation/manual-tracing/fluent-apis#decorator)를 참조하세요.

- **태깅(Tagging)**: 태그(Tags)는 트레이스의 수명 주기 동안 언제든 업데이트할 수 있는 유연한 키-값 쌍인 반면, 메타데이터(Metadata)는 변경할 수 없으며 트레이스 생성 시점에 한 번만 설정됩니다.
![tagging-example.png](../Includes/images/tagging-example.png "tagging-example.png")
<p>
<em>
태그가 MLflow 인터페이스에서 어떻게 나타나는지 보여주는 예시입니다.
</em>
</p>

> 이 과정에서는 Span 객체 스키마의 일부 하위 집합만을 다룹니다. Span 객체 스키마에 대한 자세한 내용은 [여기](https://mlflow.org/docs/latest/genai/concepts/span/#span-object-schema)에서 확인할 수 있습니다.

## C. 에이전트 거버넌스를 위한 Unity catalog 모델

MLflow와 **Unity Catalog** 의 통합은 에이전트를 **Unity Catalog 모델** 로 등록하여, 다른 비즈니스 핵심 자산과 동일하게 엄격한 기준으로 관리할 수 있도록 지원함으로써 에이전트 배포에 기업 특화형(enterprise-grade) 거버넌스를 제공합니다.


![mlflow-with-uc-diagram.png](../Includes/images/mlflow-with-uc-diagram.png "mlflow-with-uc-diagram.png")
<p>
<em>
데이터 소스와 도구(또는 외부 도구)를 Unity Catalog에 등록하고 나면, 먼저 에이전트를 MLflow로 패키징한 후 해당 모델의 URI를 사용하여 에이전트 코드를 UC에 등록할 수 있습니다.
</em>
</p>


### UC의 Model Registry를 통한 중앙집중식 거버넌스

에이전트를 UC 모델로 등록하면 여러 워크스페이스에 걸쳐 에이전트 자산을 중앙에서 통합 관리할 수 있는 카탈로그가 제공됩니다.
- **버전 관리**: MLflow는 특정 시점의 에이전트 코드, 설정 및 선언된 리소스의 스냅샷을 기록합니다. 각 UC 모델 버전은 참조 및 배포가 가능한 불변(immutable)의 스냅샷입니다.
- **계보(Lineage) 추적**: 입력 데이터를 기록할 때(예: `mlflow.log_input` 사용 시), UC는 모델과 상위 데이터 세트 간의 계보를 보여줍니다. 기능 저장소(Feature Store) 학습 흐름에 대해서도 계보가 캡처됩니다.
- **접근 제어**: 세분화된 UC 권한 설정을 통해 모델의 생성, 조회, 수정 권한은 물론, 에이전트가 의존하는 함수 실행, 테이블 조회, 연결(connection) 사용 및 기타 리소스에 접근할 수 있는 권한을 통제합니다.
- **워크스페이스 간 공유**:동일한 메타스토어에 연결된 여러 워크스페이스 전체에서 UC 내의 모델을 검색하고 관리할 수 있습니다.
- **거버넌스 태그**: 등록된 모델 및 모델 버전에 태그를 적용할 수 있습니다. 거버넌스 태그(공개 미리 보기)는 일관된 분류와 통제를 위해 표준화된 키/값 및 할당 권한을 강제합니다. 자세한 내용은 관련 문서를 참조하세요. 문서를 [여기](https://docs.databricks.com/aws/en/database-objects/tags#supported-securable-objects)에서 확인하세요.

### 재현 가능한 배포

UC와 MLflow를 함께 사용하면 에이전트 배포의 재현 가능성과 관찰 가능성을 보장할 수 있습니다.
- **불변의 버전**: 등록된 모델 버전은 변경 불가능한 스냅샷입니다. 필요한 경우 메타데이터는 업데이트할 수 있지만, 코드나 종속성을 변경하려면 새로운 버전을 생성해야 합니다.
- **종속성 캡처**: 일관된 로딩과 서빙을 보장하기 위해 MLflow는 환경 종속성(예: pip/conda를 통해)을 자동으로 캡처합니다.
- **관리형 서빙**: UC에 등록된 에이전트를 모델 서빙(Model Serving) 엔드포인트에 배포할 수 있으며, 여기에는 자체적인 스케일링, 트레이싱 기능과 피드백 및 모니터링을 위한 검토 앱(Review Apps)이 내장되어 있습니다.

## 결론

MLflow는 초기 실험 단계부터 운영 배포 및 모니터링에 이르기까지 AI 에이전트 개발의 모든 측면을 아우르는 종합적인 플랫폼으로 진화했습니다. 실험 추적, 트레이싱, 모델 레지스트리, 평가 기능의 결합은 현대 AI 에이전트의 복잡성을 다루기에 유일무이할 정도로 최적화되어 있습니다.

이 플랫폼이 Unity Catalog 및 광범위한 Databricks 에코시스템과 통합됨으로써, 기업형 에이전트 배포에 필수적인 거버넌스, 보안, 확장성을 제공합니다. 비즈니스 애플리케이션에서 AI 에이전트의 중요성이 날로 커짐에 따라, 에이전트 개발의 기반 플랫폼으로서 MLflow의 역할은 앞으로도 계속 확대될 것입니다.

## 다음 단계
이제 트레이싱, 태깅, 그리고 Unity Catalog를 활용한 재현 가능한 에이전트 구축을 위한 MLflow의 기본적인 개념을 이해하셨으므로, MLflow를 사용한 트레이싱을 다루는 다음 데모를 진행할 준비가 되었습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>